# 20 — Mathematical Foundations Capstone

**Master syllabus coverage:** V2 2.15

## Why this matters

A complete differentiable classifier integrates tensors, matrix multiplication, affine maps, probability, stable cross-entropy, chain-rule gradients, numerical verification, and optimization in one visible system.

## Learning objectives

- Implement stable multiclass softmax regression entirely in NumPy.
- Derive all forward and backward tensor shapes.
- Gradient-check every trainable value against finite differences.
- Optimize on training data, evaluate held-out data, and verify against PyTorch.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## Capstone contract

Build multiclass softmax regression from NumPy primitives. Derive every tensor shape,
implement stable cross-entropy, backpropagate analytically, gradient-check parameters,
optimize with gradient descent, and interpret uncertainty. This integrates V2 Part II
without relying on PyTorch autograd.

Shapes: examples $X=[N,D]$, weights $W=[D,K]$, bias $b=[K]$, logits and probabilities
$[N,K]$, integer targets $[N]$.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

rng = np.random.default_rng(42)
examples_per_class, dimensions, classes = 60, 2, 3
centers = np.array([[-2.0, -1.5], [2.0, -1.0], [0.0, 2.0]])
X = np.concatenate([
    rng.normal(loc=center, scale=0.7, size=(examples_per_class, dimensions))
    for center in centers
]).astype(np.float64)
targets = np.repeat(np.arange(classes), examples_per_class)
permutation = rng.permutation(len(X))
X, targets = X[permutation], targets[permutation]

split = 140
train_x, val_x = X[:split], X[split:]
train_y, val_y = targets[:split], targets[split:]
print("train/validation:", train_x.shape, val_x.shape, train_y.shape, val_y.shape)


## 1. Forward pass and stable loss

$$Z=XW+b,\qquad P_{ik}=\frac{e^{Z_{ik}-m_i}}{\sum_j e^{Z_{ij}-m_i}}$$

Mean cross-entropy is $L=-\frac1N\sum_i\log P_{i,y_i}$. Subtracting the per-row
maximum preserves softmax while preventing exponent overflow.


In [ ]:
def forward(x: np.ndarray, weight: np.ndarray, bias: np.ndarray):
    logits = x @ weight + bias
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    probabilities = np.exp(log_probabilities)
    return logits, log_probabilities, probabilities

def loss_and_gradients(x, y, weight, bias):
    logits, log_probabilities, probabilities = forward(x, weight, bias)
    loss = -log_probabilities[np.arange(len(x)), y].mean()

    dlogits = probabilities.copy()
    dlogits[np.arange(len(x)), y] -= 1.0
    dlogits /= len(x)
    dweight = x.T @ dlogits
    dbias = dlogits.sum(axis=0)
    return loss, dweight, dbias, logits, probabilities

weight = rng.normal(scale=0.01, size=(dimensions, classes))
bias = np.zeros(classes)
initial = loss_and_gradients(train_x, train_y, weight, bias)
print("initial loss:", initial[0], "uniform baseline:", np.log(classes))
np.testing.assert_allclose(initial[4].sum(axis=1), 1.0)


## 2. Backward derivation

Softmax-cross-entropy gives $dZ=(P-Y)/N$. Matrix calculus then gives

$$dW=X^\top dZ,\qquad db=\sum_i dZ_i,\qquad dX=dZW^\top.$$

Every derivative has the same shape as its differentiated quantity. Bias gradients sum
across the broadcast example axis.


In [ ]:
loss, dweight, dbias, logits, probabilities = initial
assert dweight.shape == weight.shape and dbias.shape == bias.shape
dlogits = probabilities.copy()
dlogits[np.arange(len(train_x)), train_y] -= 1
dlogits /= len(train_x)
dx = dlogits @ weight.T
assert dx.shape == train_x.shape
print("gradient shapes:", dweight.shape, dbias.shape, dx.shape)


## 3. Gradient check before optimization

Central finite differences provide an implementation independent of the analytical
backward formulas. Use float64, deterministic inputs, and a moderate epsilon. Check both
weight and bias rather than assuming one passing element proves the entire graph.


In [ ]:
def finite_difference(parameter, index, evaluate, epsilon=1e-6):
    original = parameter[index]
    parameter[index] = original + epsilon; plus = evaluate()
    parameter[index] = original - epsilon; minus = evaluate()
    parameter[index] = original
    return (plus - minus) / (2 * epsilon)

evaluate = lambda: loss_and_gradients(train_x, train_y, weight, bias)[0]
for index in np.ndindex(weight.shape):
    numeric = finite_difference(weight, index, evaluate)
    np.testing.assert_allclose(numeric, dweight[index], rtol=1e-6, atol=1e-8)
for index in np.ndindex(bias.shape):
    numeric = finite_difference(bias, index, evaluate)
    np.testing.assert_allclose(numeric, dbias[index], rtol=1e-6, atol=1e-8)
print("all weight and bias gradients passed finite differences")


## 4. Optimize and evaluate separately

The training split updates parameters. The validation split estimates behavior on held-out
examples and must not influence gradients. Track loss and accuracy, and stop immediately
on non-finite values.


In [ ]:
learning_rate = 0.2
history = []
for step in range(201):
    loss, dweight, dbias, _, _ = loss_and_gradients(train_x, train_y, weight, bias)
    assert np.isfinite(loss) and np.isfinite(dweight).all() and np.isfinite(dbias).all()
    weight -= learning_rate * dweight
    bias -= learning_rate * dbias
    history.append(loss)

def evaluate_split(x, y):
    _, log_probabilities, probabilities = forward(x, weight, bias)
    loss = -log_probabilities[np.arange(len(x)), y].mean()
    accuracy = np.mean(probabilities.argmax(axis=1) == y)
    return float(loss), float(accuracy)

train_metrics = evaluate_split(train_x, train_y)
validation_metrics = evaluate_split(val_x, val_y)
print("train loss/accuracy:", train_metrics)
print("validation loss/accuracy:", validation_metrics)
assert history[-1] < history[0] * 0.2 and validation_metrics[1] > 0.9


## 5. Framework verification and uncertainty

PyTorch should reproduce logits and loss for the same parameters. Predicted probability is
model confidence under its fitted assumptions, not guaranteed real-world correctness or
calibration. Inspect ambiguous points rather than treating argmax as the whole model output.


In [ ]:
torch_logits = torch.tensor(val_x) @ torch.tensor(weight) + torch.tensor(bias)
torch_loss = F.cross_entropy(torch_logits, torch.tensor(val_y))
numpy_logits, _, numpy_probabilities = forward(val_x, weight, bias)
np.testing.assert_allclose(torch_logits.numpy(), numpy_logits, rtol=1e-12)
np.testing.assert_allclose(torch_loss.item(), validation_metrics[0], rtol=1e-12)

confidence = numpy_probabilities.max(axis=1)
least_confident = np.argsort(confidence)[:5]
for index in least_confident:
    print("x=", val_x[index].round(2), "target=", val_y[index],
          "probabilities=", numpy_probabilities[index].round(3))


## Failure modes to recognize

- **Unstable exponentials:** otherwise valid large logits produce NaN.
- **Missing mean or broadcast reduction:** analytical and numerical gradients disagree.
- **Training/validation leakage:** held-out metrics influence parameter updates.
- **Confidence overclaim:** softmax probability is presented as guaranteed correctness.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Re-derive the gradients without reading the notebook and annotate every matrix product.
2. Add L2 regularization and gradient-check the changed objective.
3. Replace plain gradient descent with your manual momentum and Adam implementations.
4. Write a short report connecting each Part II module to one line of this capstone.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can rebuild, derive, gradient-check, optimize, and evaluate this classifier without autograd or copied formulas.

**Next:** 21 — Learning from data.
